### This Notebook studies changes in mobility patterns in Castilla y León between 2023 and 2024.
### One week in October 2023 is compared with an analogue week in 2024. The chosen week avoids bank holidays in both years.

In [1]:
import os
import pandas as pd
import glob
from datetime import datetime
import tarfile
import shutil
import gzip

### Definitions

In [2]:
# Function to create the trip matrix with the desired structure
def map_residence(value):
	
	CyL_provinces = {
	'24': 'León',
	'34': 'Palencia',
	'09': 'Burgos',
	'42': 'Soria',
	'40': 'Segovia',
	'05': 'Ávila',
	'37': 'Salamanca',
	'49': 'Zamora',
	'47': 'Valladolid'
	}

	if pd.isna(value):
		return 'Unknown'
	for code in CyL_provinces.keys():
		if value.startswith(code):
			return CyL_provinces[code]

	return 'resto de España y extranjero'

# Function to aggregate days
def aggregate_days(filtered_folder, criteria, output_file,):
	# Get a list of all CSV files in the folder
	file_list = glob.glob(os.path.join(filtered_folder, "*.csv"))
	# csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
	
	# Initialize an empty list to store data frames
	data_list = []
	
	# Loop through each CSV file
	for file in file_list:
		# Extract the date part from the filename assuming format YYYYMMDD
		file_name = os.path.basename(file)
		date_str = file_name[:8]  # Assuming the first 8 characters are the date
		# year_month = date_str[:6]  # Extract YYYYMM
		
		# Convert thhe date_str to a datetime object
		file_date = datetime.strptime(date_str, "%Y%m%d")

		# Create a weekday flag (Monday=0, Sunday=6)
		weekday_flag = 1 if file_date.weekday() < 5 else 0

		# Read the data
		data = pd.read_csv(file, dtype=data_types, sep="|")
		data['date_str'] = date_str
		data['weekday_flag'] = weekday_flag
		
		trips_before = data['viajes'].sum()
		
		# Simplify the dataframe by simplifying the zone structure
		
		# Create simpler zone structure for origin and destination
		data['origin_gr'] = data['origen'].apply(map_residence)
		data['dest_gr'] = data['destino'].apply(map_residence)

		grouping_criteria = ['origin_gr', 'dest_gr', 'date_str', 'weekday_flag'] + criteria[2:]

		# The following line fills any NaN before grouping to prevent rows from being lost
		data[grouping_criteria] = data[grouping_criteria].fillna("Unknown")
		grouped = data.groupby(grouping_criteria).agg(
			viajes=('viajes', 'sum'),
			viajes_km=('viajes_km', 'sum')
		).reset_index()

		# Print the total number of trips each day
		trips_after = grouped['viajes'].sum()
		assert round(trips_before,4) == round(trips_after,4), (
			f"Trips before and after simplification do not match: {trips_before} vs {trips_after}"
			)
		
		data_list.append(grouped)
		print(f"{os.path.basename(file)} completed. Total trips added: {trips_after}")

	# Combine all data frames into one
	combined_data = pd.concat(data_list, ignore_index=True)
	print(f"Total trips: {combined_data['viajes'].sum()}")

	print(f"Output trips({output_file}): {combined_data['viajes'].sum()}")
	print(f"Output trip-kilometers({output_file}): {combined_data['viajes_km'].sum()}")

	# Write the filtered data to a new CSV file
	combined_data.drop_duplicates(inplace = True)
	combined_data.to_csv(os.path.join(output_folder, output_file), index=False)

### Pseudo main

In [3]:
# Define working folder. We'll work in parallel with the 3 possible subdivisions (GAU, municipality, census district)
working_folder = "D:/Documents/Proyectos/MITMA/Data"

# Define sub-folders
tar_file_path = os.path.join(working_folder, "0-meses_completos/")
raw_files = os.path.join(working_folder, "1-Unzipped/")
filtered_folder = os.path.join(working_folder, "2-CyL/")

if os.path.exists(filtered_folder):
	print("The region you have selected already contains results. The execution will delete the existing results")
	# if input("Are you sure you want to continue? (y/n)") != "y":
	# 	exit()
	shutil.rmtree(filtered_folder)
else:
	os.mkdir(filtered_folder)

output_folder = os.path.join(working_folder,"3-Results/")
zones_shp = os.path.join(working_folder, "Zonificación_GAUs/")
# output_folder = os.path.join(working_folder, "3-Averages/")

# Create a temporary folder to store the .gz files inside each .tar file
tmp_folder = os.path.join(working_folder, "1-tmp")
if os.path.exists(tmp_folder):
	shutil.rmtree(tmp_folder)
os.makedirs(tmp_folder)
print("Completed restart of temp folder")

# Define datatypes
data_types = {"origen": 'string',
			  "destino": 'string', 
			  "distancia": 'string',
			  "actividad_origen": 'string',
			  "actividad_destino": 'string',
			  "residencia": 'string',
			  "renta": 'string',
			  "edad": 'string',
			  "sexo": 'string'
}

criteria = list(data_types.keys())

region = "CyL"

# Define the date to filter
date_list = ['20231016', '20231017', '20231018', '20231019', '20231020', '20231021', '20231022',
			 '20241021', '20241022', '20241023', '20241024', '20241025', '20241026', '20241027'
			 ]

# Define the tar files to process
tar_file_list = ['202310_Viajes_distritos.tar', '202410_Viajes_distritos.tar']

print(f"Tar files to be processed: {tar_file_list}")
pattern = os.path.join(raw_files, "*.csv")
file_list = glob.glob(pattern)
output_file = region + "_MITMA_2023.csv"

# Apply the filter function to each .tar file
count = 0
for tar_file in tar_file_list:
	tar_file = os.path.join(tar_file_path, tar_file)
	with tarfile.open(tar_file, 'r') as tar:
		for daily_file in tar.getmembers():
			if os.path.basename(daily_file.name)[:8] in date_list:
				count += 1
				print(f"Extracting {daily_file.name}. Total files extracted: {count}")
				tar.extract(daily_file, path=tmp_folder)
				gz_file_path = os.path.join(tmp_folder, daily_file.name)
				with gzip.open(gz_file_path, 'rb') as f_in:
					with open(gz_file_path[:-3], 'wb') as f_out:
						shutil.copyfileobj(f_in, f_out)
			else:
				print(f"Skipping {daily_file}. Date not in list")

print("Completed extraction of files")

aggregate_days(tmp_folder, criteria, output_file)


The region you have selected already contains results. The execution will delete the existing results
Completed restart of temp folder
Tar files to be processed: ['202310_Viajes_distritos.tar', '202410_Viajes_distritos.tar']
20231016_Viajes_distritos.csv completed. Total trips added: 140329034.57999998
20231017_Viajes_distritos.csv completed. Total trips added: 141988978.65100002
20231018_Viajes_distritos.csv completed. Total trips added: 142856559.359
20231019_Viajes_distritos.csv completed. Total trips added: 138867557.08699998
20231020_Viajes_distritos.csv completed. Total trips added: 144744335.75300002
20231021_Viajes_distritos.csv completed. Total trips added: 123367553.30399999
20231022_Viajes_distritos.csv completed. Total trips added: 104584884.36999999
20241021_Viajes_distritos.csv completed. Total trips added: 146904068.692
20241022_Viajes_distritos.csv completed. Total trips added: 147973397.15
20241023_Viajes_distritos.csv completed. Total trips added: 148427847.14000002
2